# NPS with label clouds and a position-only sampler

This **synthetic** continuation of `nps.ipynb` illustrates three distinct
BFF operations: (i) a numerical cloud-average of the NPS FRET forward model,
(ii) a separately evaluated KDE *position prior*, and (iii) an MCMC network
example with **explicit proper Gaussian coordinate priors**. Neither the
accessible-volume cloud nor its KDE prior is wired into the sampler
implicitly. Angles are fixed by `NPSNetworkObjective`; the orientation-aware
posterior of the original NPS studies requires extra degrees of freedom [1, 2].

Fast-NPS was published by **Eilert, Beckers, Drechsler & Michaelis (2017)**,
not by Muschielok & Michaelis as an alleged 2008/2017 CPC pair [2]. BFF's
caller-provided clouds, KDE prior, and scaled polynomial fit are BFF-specific
extensions; they are not claims of bitwise agreement with Fast-NPS [2, 3].
All coordinates are in Å; the toy observations below are not experimental data.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

import IMP.bff as bff

## 1. Local dye-position clouds

Each cloud is a **flat** `(n, 4)` sequence of local $(x,y,z,w)$ offsets from
its label attachment site; weights are non-negative. We sample uniform
spherical toy clouds with explicitly seeded random generators. An actual
accessible-volume cloud must reflect steric and linker chemistry rather than
be assumed spherical [1, 3]. For position priors below we additionally make
an absolute-frame cloud around the acceptor attachment site.

In [ ]:
def cloud_sphere(center, radius, n=2500, seed=0):
    """Draw equally weighted, uniformly distributed points inside a toy sphere."""
    rng = np.random.default_rng(seed)
    center = np.asarray(center, dtype=float)
    pts = rng.uniform(-radius, radius, size=(n * 3, 3)) + center
    inside = np.sum((pts - center) ** 2, axis=1) < radius ** 2
    pts = pts[inside][:n]
    if len(pts) != n:
        raise RuntimeError("not enough in-sphere draws; increase the proposal batch")
    return np.column_stack((pts, np.full(n, 1.0 / n))).ravel()

cloud_d = cloud_sphere([0, 0, 0], 15.0, seed=1)
cloud_a = cloud_sphere([45, 0, 0], 15.0, seed=2)
assert len(cloud_d) == len(cloud_a) == 4 * 2500
print('donor/acceptor toy cloud points:', len(cloud_d) // 4, len(cloud_a) // 4)

## 2. Distance-convolved efficiency from local clouds

NPS uses $E(r)=R_{0,\mathrm{iso}}^6/(R_{0,\mathrm{iso}}^6+r^6)$ for a fully
isotropic pair; averaging $E(r)$ over dye-position clouds does **not**
generally give $E$ at the mean position [1, 2]. BFF's table generator draws
weighted pairs from caller-provided **local** clouds, evaluates 150
site-separation bins, and fits a degree-11 polynomial in $d/150$ (stored
as 12 power-basis coefficients). This is a **numerical approximation** with
finite Monte-Carlo and polynomial error, not an exact analytic convolution
or the original Fast-NPS hardcoded 20 Å sphere [2]. At $d>150$ Å the
compatibility evaluator returns zero.

For the generator only, `dep=1.0` selects an isotropically averaged table
layout (1 row). In the **direct** `NPSNetworkDye` model $q=1$ means
$r_{\mathrm{avg}}=0.4$ (rigid), so do not confuse the generator's layout
selector with a physical claim about direct anisotropy. Here the two
local clouds have the same spherical toy shape.

In [ ]:
R0 = 55.0
r_iso6 = R0 ** 6
av = cloud_sphere([0, 0, 0], 15.0, seed=1)  # local offsets

# 1 / 25 / 625 rows encode iso/iso, one orientation grid, or two grids.
table_iso = bff.nps_convolved_efficiency_table(
    av, av, 1.0, 1.0, r_iso6, n_samples=100_000, seed=3)
table_mixed = bff.nps_convolved_efficiency_table(
    av, av, 0.9, 1.0, r_iso6, n_samples=30_000, seed=3)
table_grid = bff.nps_convolved_efficiency_table(
    av, av, 0.9, 0.9, r_iso6, n_samples=8_000, seed=3)
assert [len(table_iso), len(table_mixed), len(table_grid)] == [1, 25, 625]
assert all(len(row) == 12 for table in (table_iso, table_mixed, table_grid)
           for row in table)
print('iso/iso, mixed, grid rows:', len(table_iso), len(table_mixed), len(table_grid))

# Independent stochastic check for the fully isotropic row at d=55 A.
# These toy clouds have equal weights. A physical cloud can have unequal weights.
rng_check = np.random.default_rng(12)
offsets = np.asarray(av).reshape(-1, 4)[:, :3]
index_d = rng_check.integers(len(offsets), size=25_000)
index_a = rng_check.integers(len(offsets), size=25_000)
r_pairs = np.linalg.norm([55.0, 0.0, 0.0] + offsets[index_a] - offsets[index_d],
                         axis=1)
e_direct_mc = np.mean(1.0 / (1.0 + (r_pairs / R0) ** 6))
e_table = bff.nps_convolved_efficiency(55.0, table_iso[0])
assert abs(e_table - e_direct_mc) < 0.02  # independent MC + polynomial error
print(f'isotropic cloud E(55 A): table={e_table:.4f}, independent MC={e_direct_mc:.4f}')

The coefficient rows represent the site-separation function $\bar E(d)$.
For an orientation-grid dye the evaluator selects a **single** nearest
legacy grid slot; it does not interpolate between rows [2]. In the plot,
$m=-\cos\theta$ and $\phi$ are fixed illustrative slot requests. Plot
efficiency using `nps_convolved_efficiency`, the same public evaluator
used by the likelihood, rather than silently replacing its behavior with
`numpy.polyval`.

In [ ]:
xs = np.arange(1, 151, dtype=float)
row_mixed = bff.nps_single_orientation_row_index(0.2, 0.4)
row_grid = bff.nps_orientation_row_index(0.2, 0.4, -0.2, 0.8)
print('example orientation rows (mixed, grid):', row_mixed, row_grid)
fig, ax = plt.subplots(figsize=(6, 3.2), dpi=110)
for label, table, row, ls in [
    ('isotropic pair, row 0', table_iso, 0, '-'),
    ('one oriented dye', table_mixed, row_mixed, '--'),
    ('two oriented dyes', table_grid, row_grid, ':'),
]:
    curve = np.array([bff.nps_convolved_efficiency(float(x), table[row])
                      for x in xs])
    assert np.all(np.isfinite(curve))
    ax.plot(xs, curve, ls, label=label, lw=1.8)
ax.set_xlabel('attachment-site separation d [Å]')
ax.set_ylabel('cloud-convolved mean efficiency')
ax.set_title('NPS table approximations over their fitted 1–150 Å range')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## 3. A separate KDE position prior

`nps_cloud_log_prior` evaluates a weighted, isotropic Gaussian KDE from an
**absolute-frame** site cloud. It is a normalized **3D** density
$p(\mathbf x)$ in Å$^{-3}$, not a normalized one-dimensional density along
the plotted $x$ slice. This smooth choice is a **BFF modeling extension**;
it replaces the hard box membership used by Fast-NPS, not a paper-prescribed
NPS prior [2, 3]. In the current BFF implementation the scalar bandwidth is
$h=\sqrt{\sum_i w_i\|\mathbf x_i-\bar{\mathbf x}\|^2/\sum_i w_i}\,n^{-1/5}$.
The $n^{-1/5}$ exponent is the BFF implementation's legacy heuristic;
**3D Scott scaling would use $n^{-1/7}$**, so this notebook does not label
it Scott's 3D rule. The two should not be interchanged without validation.

In [ ]:
h = bff.nps_cloud_bandwidth(cloud_a)
mean = np.asarray(bff.points_weighted_mean(cloud_a))
# Independently pin the *implemented* bandwidth, not a textbook 3D rule.
xyz = np.asarray(cloud_a).reshape(-1, 4)[:, :3]
h_implemented = np.sqrt(np.mean(np.sum((xyz - mean) ** 2, axis=1))) * len(xyz) ** (-1 / 5)
assert np.isclose(h, h_implemented, rtol=1e-12)
print(f'BFF KDE bandwidth: {h:.2f} Å; cloud mean {np.round(mean, 1)} Å')

grid = np.linspace(0, 90, 240)
log_slice = np.array([bff.nps_cloud_log_prior(cloud_a, x, 0.0, 0.0)
                      for x in grid])
fig, ax = plt.subplots(figsize=(6, 3.0), dpi=110)
ax.plot(grid, np.exp(log_slice), lw=1.8)
ax.axvline(mean[0], color='grey', ls=':', label='cloud mean x')
ax.set_xlabel('x [Å], at y=z=0')
ax.set_ylabel('3D KDE density [Å⁻³] (slice, not 1D marginal)')
ax.set_title('Absolute-frame acceptor position prior')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### Toy Metropolis walk targeting only the KDE

`nps_cloud_prior_score` is $-\lambda\log p_{\rm KDE}(\mathbf x)$. Here
$\lambda=1$ and the Metropolis temperature is **$kT=1$**, so the invariant
target is the KDE itself. A temperature of $0.5$ would instead target
$p_{\rm KDE}^2$ and is **not** an ordinary posterior draw. This illustrative
standalone walk does not attach the KDE to the graph sampler in section 5;
its endpoint and acceptance alone do not establish convergence.

In [ ]:
rng = np.random.default_rng(1)
x = np.array([60.0, 20.0, -15.0])
start_x = x.copy()
score = bff.nps_cloud_prior_score(cloud_a, *x, 1.0)
accepted = 0
for _ in range(20_000):
    proposal = x + rng.normal(0.0, 2.0, 3)
    proposed_score = bff.nps_cloud_prior_score(cloud_a, *proposal, 1.0)
    if proposed_score <= score or rng.random() < math.exp(score - proposed_score):
        x, score = proposal, proposed_score
        accepted += 1
print(f'KDE-only walk: {np.round(start_x, 1)} -> {np.round(x, 1)} Å; '
      f'acceptance={accepted/20_000:.2f}')
print(f'final distance from cloud mean: {np.linalg.norm(x-mean):.1f} Å '
      '(one seeded draw, not a localization precision)')

## 4. Seed a network configuration

`nps_config_from_cloud` uses each absolute-frame cloud's weighted mean
for $(x,y,z)$ and seeds $(m,\phi)=(0,0)$. Those angles are **not** an
orientational posterior: the position-only objective freezes them.
For the *direct* example in section 5 we set both dyes' physical
`dep=0` (fully depolarized, $\overline\kappa^2=2/3$), the condition under
which frozen orientations cannot change direct FRET [1, 2].

In [ ]:
row_d = bff.nps_config_from_cloud(cloud_d)
row_a = bff.nps_config_from_cloud(cloud_a)
config = [row_d, row_a]
print('donor row   :', np.round(row_d, 2))
print('acceptor row:', np.round(row_a, 2))

dyes = []
for _ in range(2):
    dye = bff.NPSNetworkDye()
    dye.dep, dye.iso, dye.dist_conv = 0.0, True, False
    dyes.append(dye)
e_direct_seed = bff.nps_network_fret_efficiency(config, dyes, 0, 1, r_iso6)
site_sep = np.linalg.norm(np.asarray(row_a[:3]) - np.asarray(row_d[:3]))
assert np.isclose(e_direct_seed, 1.0/(1.0 + (site_sep/R0)**6), rtol=1e-12)
print(f'direct point efficiency at cloud means: {e_direct_seed:.4f}')

## 5. A **proper** position-only posterior

The sampler has six scalar ports $(x_D,y_D,z_D,x_A,y_A,z_A)$. It does
**not** sample $(m,\phi)$; an oriented dye or active transfer anisotropy
with $q_Dq_A>0$ is rejected by `NPSNetworkObjective` at construction.
A single FRET efficiency constrains mostly the **radial separation**,
not six individual Cartesian coordinates [1, 2, 4].

We put an independent **proper Gaussian prior** with 18 Å standard deviation
on **each** coordinate port, centered at the two *synthetic cloud means*.
That makes the posterior normalizable and removes the unbounded common
translation mode. These factorized Gaussian priors are **approximations**;
they are NOT `nps_cloud_log_prior` (which would require an explicit objective
composition). The prior width is a toy choice, not measured linker
uncertainty. The log likelihood and all six log priors are added by
`MCMCSampler` [2, 3].

In [ ]:
import json

meas = bff.NPSMeasurement()
meas.dye1, meas.dye2 = 0, 1
meas.fret_active = True
meas.e_avg, meas.e_err = 0.6, 0.05  # illustrative synthetic observation
meas.r_iso6 = r_iso6

node = bff.NPSNetworkObjective(dyes, [meas], 2)
prior_centers = np.r_[row_d[:3], row_a[:3]].astype(float)
prior_sigma = 18.0  # Å, illustrative weak coordinate priors
ports = []
for i, center in enumerate(prior_centers):
    port = bff.GraphPort([float(center)])
    port.set_prior(json.dumps({'kind': 'normal', 'mu': float(center),
                               'sigma': prior_sigma}))
    ports.append(port)
    link = bff.GraphPort([0.0])
    link.set_link(port)
    node.add_input_port(f'x{i}', link)
node.add_output_port('chi2', bff.GraphPort([0.0]))
node.set_output_is_log_likelihood(True)

r_opt = R0 * ((1.0 - meas.e_avg) / meas.e_avg) ** (1.0 / 6.0)
assert np.isclose(1.0 / (1.0 + (r_opt/R0)**6), meas.e_avg, atol=1e-12)
print(f'likelihood-only radial mode: {r_opt:.1f} Å (not the full posterior mode)')

In [ ]:
sampler = bff.MCMCSampler('stretch', 11)
sampler.set_parameter_ports(ports)
sampler.set_objective(node, 'chi2')
sampler.set_output_is_log_likelihood(True)

rng = np.random.default_rng(4)
start = rng.normal(prior_centers, 0.5, size=(16, 6)).tolist()
for row in start:
    row[3] += r_opt - site_sep
sampler.set_walker_start(start)
sampler.run(400)  # a short teaching run; never call this a converged analysis

chain = np.asarray(sampler.get_chain())
ln_prior = np.asarray(sampler.get_lnprior())
assert chain.ndim == 2 and chain.shape[1] == 6
assert len(ln_prior) == len(chain) and np.all(np.isfinite(ln_prior))
assert np.std(ln_prior) > 0.0
print('proper prior: six independent Normal coordinate ports; '
      f'ln prior range [{ln_prior.min():.1f}, {ln_prior.max():.1f}]')
print(f'chain rows: {chain.shape}; acceptance: {sampler.get_acceptance_rate():.2f}')

radial = np.linalg.norm(chain[:, 3:6] - chain[:, :3], axis=1)
assert np.all(np.isfinite(radial))
# This deterministic smoke check would fail if the sampler preferred
# *worse* FRET fits (for example from a reversed log-L port).
assert abs(np.median(radial) - r_opt) < 12.0
print(f'radial separation: median {np.median(radial):.1f} Å; '
      f'16–84% of recorded draws {np.quantile(radial, [0.16, 0.84]).round(1)} Å')
fig, ax = plt.subplots(figsize=(6, 3.0), dpi=110)
ax.hist(radial, bins=40, density=True, color='#4878cf', alpha=0.85)
ax.axvline(r_opt, color='k', lw=1, label='likelihood-only mode %.1f Å' % r_opt)
ax.set_xlabel('radial dye separation |p_A - p_D| [Å]')
ax.set_ylabel('empirical density of short-chain draws [Å⁻¹]')
ax.set_title('Toy posterior draws; convergence not established')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

The Gaussian priors make common translations proper, but one FRET datum
still leaves substantial prior-dependent angular uncertainty. The histogram
plots **radial distance**, not a signed $x_A-x_D$ coordinate. The vertical
line is the **likelihood-only** optimum
$R_{0,\mathrm{iso}}[(1-E_{\mathrm{obs}})/E_{\mathrm{obs}}]^{1/6}$ (51.4 Å);
the full posterior also depends on the stated priors. A short-chain
acceptance rate, one-walker MCSE, and the minimum of the recorded
$-2\log L$ do not prove mixing, a credible interval, or goodness of fit.
In log-likelihood mode `sampler.get_chi2()` stores $-2\log L$, **not**
the non-negative squared-residual $\chi^2$: normalized Gaussian
densities can exceed one, so $-2\log L$ can be negative. For scientific
inference use longer independent runs and between-chain diagnostics [4, 5].

In [ ]:
ce = bff.nps_cross_entropy(sampler.get_chi2(), sampler.get_lnprior())
walker0 = np.asarray(sampler.get_chain_of_walker(0), dtype=float)
mcse_xyz = bff.nps_mean_mcse(walker0.tolist())
walker0_radial = np.linalg.norm(walker0[:, 3:6] - walker0[:, :3], axis=1)
mcse_radial = bff.nps_mean_mcse(walker0_radial[:, None].tolist())[0]
chi2 = np.asarray(sampler.get_chi2())
assert np.all(np.isfinite(chi2)) and np.isfinite(ce)
print(f'negative mean recorded log posterior (cross-entropy): {ce:.2f}')
print(f'one-walker coordinate MCSE range: {min(mcse_xyz):.2g}–{max(mcse_xyz):.2g} Å')
print(f'one-walker radial-mean MCSE: {mcse_radial:.2g} Å')
lower_bound = 2.0 * math.log(meas.e_err * math.sqrt(2.0 * math.pi))
assert chi2.min() >= lower_bound - 1e-10
print(f'minimum recorded -2 log L: {chi2.min():.3f}; '
      f'Gaussian lower bound: {lower_bound:.3f} (this is not χ²)')

## 6. Generated table inside the likelihood

To show the *different* position-cloud forward model end to end, turn on
`dist_conv` for both dyes and set `iso=True` to select **row 0** of the
one-row table. The likelihood then evaluates at the distance between the
attachment-site mean coordinates, while the table averages over local
cloud offsets. `nps_network_log_likelihood` here evaluates one **fixed**
configuration and is **not** a chain from section 5. Distinguish the
point-position, convolution and coordinate-prior ingredients [1, 2, 3].

In [ ]:
meas_grid = bff.NPSMeasurement()
meas_grid.dye1, meas_grid.dye2 = 0, 1
meas_grid.fret_active = True
meas_grid.e_avg, meas_grid.e_err = 0.55, 0.05
meas_grid.r_iso6 = r_iso6
meas_grid.set_eff_conv_coeff(table_iso)

dyes_conv = []
for _ in range(2):
    dye = bff.NPSNetworkDye()
    dye.dep, dye.iso, dye.dist_conv = 0.0, True, True
    dyes_conv.append(dye)

e_conv = bff.nps_network_fret_efficiency(config, dyes_conv, 0, 1, r_iso6,
                                         table_iso)
assert np.isclose(e_conv, bff.nps_convolved_efficiency(site_sep, table_iso[0]),
                  rtol=1e-12)
logl = bff.nps_network_log_likelihood(config, dyes_conv, [meas_grid])
manual = (-0.5 * ((meas_grid.e_avg - e_conv) / meas_grid.e_err) ** 2
          - math.log(meas_grid.e_err * math.sqrt(2.0 * math.pi)))
assert np.isclose(logl, manual, rtol=1e-12)
print(f'cloud-averaged E at site separation {site_sep:.1f} Å: {e_conv:.4f}')
print(f'point-position E at same mean separation: {e_direct_seed:.4f}')
print(f'one-measurement normalized network log L: {logl:.4f}')

## Primary references and implementation caveats

[1] Muschielok A & Michaelis J. **Application of the Nano-Positioning System
to the Analysis of Fluorescence Resonance Energy Transfer Networks.**
*J. Phys. Chem. B* **115**, 11927–11937 (2011).
doi:10.1021/jp2060377. Its supplementary theory describes an orientation
prior flat in $m=-\cos\theta$ and $\phi$; our position-only toy is not a
replication of its full orientation posterior.

[2] Eilert T, Beckers M, Drechsler F & Michaelis J. **Fast-NPS—A Markov Chain
Monte Carlo-based analysis tool to obtain structural information from
single-molecule FRET measurements.** *Comput. Phys. Commun.* **219**,
377–389 (2017). doi:10.1016/j.cpc.2017.05.027. Its released implementation
provides the 1/25/625-row behavioral comparison, not the toy KDE or BFF
polynomial fitting procedure.

[3] Beckers M *et al.* **Quantitative structural information from
single-molecule FRET.** *Faraday Discuss.* **184**, 117–129 (2015).
doi:10.1039/c5fd00110b.

[4] Muschielok A *et al.* **A nano-positioning system for macromolecular
structural analysis.** *Nature Methods* **5**, 965–971 (2008).
doi:10.1038/nmeth.1259.

[5] Vehtari A *et al.* **Rank-Normalization, Folding, and Localization: An
Improved $\hat R$ for Assessing Convergence of MCMC.** *Bayesian Analysis*
**16** (2021). doi:10.1214/20-BA1221. This supports the caution about
convergence; the plotted one-walker MCSE is not a between-chain diagnostic.

BFF's KDE bandwidth is a **legacy heuristic** (its $n^{-1/5}$ exponent
does not match isotropic 3D Scott $n^{-1/7}$). Do **not** attribute the
implemented exponent to Scott's 1992 *Multivariate Density Estimation* or
to an invalid DOI. Published methods constrain physical interpretation;
the numerical KDE and posterior choices above are explicitly ours.